In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/formula-1-race-result-classification-challenge/sample_submission.csv
/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv
/kaggle/input/competitions/formula-1-race-result-classification-challenge/test.csv


In [2]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/formula-1-race-result-classification-challenge/sample_submission.csv
/kaggle/input/competitions/formula-1-race-result-classification-challenge/train.csv
/kaggle/input/competitions/formula-1-race-result-classification-challenge/test.csv


In [3]:
import pandas as pd
import numpy as np

PATH = '/kaggle/input/competitions/formula-1-race-result-classification-challenge/'

train = pd.read_csv(PATH + 'train.csv')
test = pd.read_csv(PATH + 'test.csv')
sample = pd.read_csv(PATH + 'sample_submission.csv')

print("Train:", train.shape)
print("Test :", test.shape)
print("Sample:", sample.shape)

train.head()

Train: (25749, 28)
Test : (919, 27)
Sample: (919, 2)


,id,raceId,driverId,constructorId,year,round,circuitId,lat,lng,alt,...,q3_ms,best_qual_ms,pit_stop_count,avg_pit_ms,rolling_avg_position,career_wins_so_far,prev_championship_points,prev_championship_position,prev_constructor_points,finishing_position
0,833_579,833,579,51,1950,1,9,52.13126,-1.01516,153,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,12
1,833_642,833,642,51,1950,1,9,52.13126,-1.01516,153,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,1
2,833_686,833,686,51,1950,1,9,52.13126,-1.01516,153,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,3
3,833_786,833,786,51,1950,1,9,52.13126,-1.01516,153,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,2
4,833_589,833,589,105,1950,1,9,52.13126,-1.01516,153,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,18


In [4]:
import pandas as pd

PATH = '/kaggle/input/competitions/formula-1-race-result-classification-challenge/'

train = pd.read_csv(PATH + 'train.csv')
test = pd.read_csv(PATH + 'test.csv')
sample = pd.read_csv(PATH + 'sample_submission.csv')

print(train.shape, test.shape)
train.head()

(25749, 28) (919, 27)


,id,raceId,driverId,constructorId,year,round,circuitId,lat,lng,alt,...,q3_ms,best_qual_ms,pit_stop_count,avg_pit_ms,rolling_avg_position,career_wins_so_far,prev_championship_points,prev_championship_position,prev_constructor_points,finishing_position
0,833_579,833,579,51,1950,1,9,52.13126,-1.01516,153,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,12
1,833_642,833,642,51,1950,1,9,52.13126,-1.01516,153,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,1
2,833_686,833,686,51,1950,1,9,52.13126,-1.01516,153,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,3
3,833_786,833,786,51,1950,1,9,52.13126,-1.01516,153,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,2
4,833_589,833,589,105,1950,1,9,52.13126,-1.01516,153,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,18


In [5]:
# Simple baseline
sub = test[['raceId','driverId']].copy()
sub['finishing_position'] = test['grid'].clip(1, 20)

sub.to_csv('/kaggle/working/submission.csv', index=False)

print(sub.head())
print("Saved!")

   raceId  driverId  finishing_position
0    1098       846                  11
1    1098       857                  18
2    1098       848                  15
3    1098       858                  16
4    1098       832                   4
Saved!


In [6]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold

# =========================
# TARGET
# =========================
TARGET = 'finishing_position'
# 🔥 SUPER STRONG FEATURE
train['grid_weighted'] = train['grid'] * 2
test['grid_weighted'] = test['grid'] * 2

# =========================
# 🔥 FEATURE ENGINEERING (KEY BOOST)
# =========================

# Basic strong features
train['grid_squared'] = train['grid'] ** 2
test['grid_squared'] = test['grid'] ** 2

train['grid_x_form'] = train['grid'] * train['rolling_avg_position']
test['grid_x_form'] = test['grid'] * test['rolling_avg_position']

train['qualifying_gap'] = train['best_qual_ms'] - train['q1_ms']
test['qualifying_gap'] = test['best_qual_ms'] - test['q1_ms']

# 🔥 RACE-WISE FEATURES (GAME CHANGER)
train['grid_rank'] = train.groupby('raceId')['grid'].rank()
test['grid_rank'] = test.groupby('raceId')['grid'].rank()

train['qual_rank'] = train.groupby('raceId')['best_qual_ms'].rank()
test['qual_rank'] = test.groupby('raceId')['best_qual_ms'].rank()

train['form_rank'] = train.groupby('raceId')['rolling_avg_position'].rank()
test['form_rank'] = test.groupby('raceId')['rolling_avg_position'].rank()

train['grid_norm'] = train['grid'] / train.groupby('raceId')['grid'].transform('max')
test['grid_norm'] = test['grid'] / test.groupby('raceId')['grid'].transform('max')


# =========================
# FEATURES
# =========================
features = [c for c in train.columns if c not in [TARGET, 'raceId', 'driverId', 'id']]

# =========================
# DATA PREP
# =========================
X = train[features].fillna(-999)
y = train[TARGET] - train['grid']
X_test = test[features].fillna(-999)

# =========================
# MODEL PARAMS
# =========================
params = {
    'objective': 'regression_l1',
    'metric': 'mae',
    'learning_rate': 0.02,
    'num_leaves': 128,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'min_child_samples': 10,
    'verbose': -1,
}

# =========================
# CROSS VALIDATION (TRAINING)
# =========================
kf = KFold(n_splits=5, shuffle=True, random_state=42)

test_preds = np.zeros(len(X_test))
oof = np.zeros(len(X))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    print(f"Running Fold {fold}...")
    
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    
    model = lgb.LGBMRegressor(**params, n_estimators=3000)
    
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
    )
    
    oof[val_idx] = model.predict(X_val)
    test_preds += model.predict(X_test) / 5
    
    fold_mae = np.mean(np.abs(y_val - oof[val_idx]))
    print(f"Fold {fold} MAE: {fold_mae:.4f}")

print("\n✅ FINAL MAE:", np.mean(np.abs(y - oof)))

# =========================
# SAVE SUBMISSION
# =========================
sub = test[['raceId','driverId']].copy()
sub['finishing_position'] = np.clip(test['grid'] + test_preds, 1, 20)

sub.to_csv('/kaggle/working/submission.csv', index=False)

print(sub.head())
print("🔥 Saved! Now submit via Save Version")

Running Fold 0...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[292]	valid_0's l1: 4.48925
Fold 0 MAE: 4.4892
Running Fold 1...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[322]	valid_0's l1: 4.53303
Fold 1 MAE: 4.5330
Running Fold 2...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[292]	valid_0's l1: 4.54437
Fold 2 MAE: 4.5444
Running Fold 3...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[508]	valid_0's l1: 4.57821
Fold 3 MAE: 4.5782
Running Fold 4...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[292]	valid_0's l1: 4.53742
Fold 4 MAE: 4.5374

✅ FINAL MAE: 4.536452501741106
   raceId  driverId  finishing_position
0    1098       846            8.822859
1    1098       857           15.653205
2    1098       848           14.04600

In [7]:
import pandas as pd
import numpy as np
import lightgbm as lgb

# =========================
# LOAD DATA (if not already loaded)
# =========================
PATH = '/kaggle/input/competitions/formula-1-race-result-classification-challenge/'

train = pd.read_csv(PATH + 'train.csv')
test = pd.read_csv(PATH + 'test.csv')

TARGET = 'finishing_position'
# Driver vs constructor strength
train['driver_vs_constructor'] = train['prev_championship_points'] - train['prev_constructor_points']
test['driver_vs_constructor'] = test['prev_championship_points'] - test['prev_constructor_points']
# 🔥 How much better/worse driver usually finishes vs grid
train['grid_minus_form'] = train['grid'] - train['rolling_avg_position']
test['grid_minus_form'] = test['grid'] - test['rolling_avg_position']

# 🔥 Championship pressure
train['points_per_position'] = train['prev_championship_points'] / (train['prev_championship_position'] + 1)
test['points_per_position'] = test['prev_championship_points'] / (test['prev_championship_position'] + 1)

# 🔥 Constructor strength normalized
train['constructor_strength'] = train['prev_constructor_points'] / (train['round'] + 1)
test['constructor_strength'] = test['prev_constructor_points'] / (test['round'] + 1)
# =========================
# 🔥 FEATURE ENGINEERING
# =========================
for df in (train, test):
    df['grid_squared'] = df['grid'] ** 2
    df['grid_x_form'] = df['grid'] * df['rolling_avg_position']
    df['qual_gap'] = df['best_qual_ms'] - df['q1_ms']

    # Race-relative features
    df['grid_rank'] = df.groupby('raceId')['grid'].rank()
    df['qual_rank'] = df.groupby('raceId')['best_qual_ms'].rank()
    df['form_rank'] = df.groupby('raceId')['rolling_avg_position'].rank()
    df['grid_norm'] = df['grid'] / df.groupby('raceId')['grid'].transform('max')

# =========================
# USE ONLY MODERN F1 DATA
# =========================
train_recent = train[train['year'] >= 2010].copy()

# =========================
# FEATURES
# =========================
drop_cols = [TARGET, 'raceId', 'driverId', 'id']
features = [c for c in train.columns if c not in drop_cols]

X = train_recent[features].fillna(-999)
y = train_recent[TARGET]
X_test = test[features].fillna(-999)

# =========================
# MODEL PARAMS
# =========================
params = {
    'objective': 'regression_l1',
    'metric': 'mae',
    'learning_rate': 0.03,
    'num_leaves': 64,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'min_child_samples': 20,
    'verbose': -1,
}

# =========================
# TIME-BASED CV (CORRECT)
# =========================
years = sorted(train_recent['year'].unique())

test_preds = np.zeros(len(X_test))
oof = np.zeros(len(train_recent))

for i in range(len(years) - 1):
    train_years = years[:i+1]
    val_year = years[i+1]

    tr_mask = train_recent['year'].isin(train_years)
    val_mask = train_recent['year'] == val_year

    X_tr, y_tr = X[tr_mask], y[tr_mask]
    X_val, y_val = X[val_mask], y[val_mask]

    print(f"Train {train_years[0]}–{train_years[-1]} | Val {val_year}")

    model = lgb.LGBMRegressor(**params, n_estimators=1500)

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
    )

    preds_val = model.predict(X_val)
    oof[val_mask.values] = preds_val

    test_preds += model.predict(X_test) / (len(years) - 1)

    fold_mae = np.mean(np.abs(y_val - preds_val))
    print(f"MAE: {fold_mae:.4f}\n")

# =========================
# FINAL SCORE
# =========================
valid_mask = oof != 0   # rows that received predictions
final_mae = np.mean(np.abs(y.values[valid_mask] - oof[valid_mask]))
print("✅ FINAL MAE (correct):", final_mae)

# =========================
# SAVE SUBMISSION
# =========================
sub = test[['raceId','driverId']].copy()
sub['finishing_position'] = np.clip(test_preds, 1, 20)

sub.to_csv('/kaggle/working/submission.csv', index=False)

print(sub.head())
print("🔥 Submission saved! Go to Save Version → Submit")

Train 2010–2010 | Val 2011
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[289]	valid_0's l1: 3.62164
MAE: 3.6216

Train 2010–2011 | Val 2012
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[94]	valid_0's l1: 4.16204
MAE: 4.1620

Train 2010–2012 | Val 2013
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[136]	valid_0's l1: 3.47424
MAE: 3.4742

Train 2010–2013 | Val 2014
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[158]	valid_0's l1: 3.25992
MAE: 3.2599

Train 2010–2014 | Val 2015
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[112]	valid_0's l1: 3.32052
MAE: 3.3205

Train 2010–2015 | Val 2016
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[175]	valid_0's l1: 3.43241
MAE: 3.4324

Train 2010–

In [8]:
import joblib

# Save the FINAL trained model
joblib.dump(model, '/kaggle/working/f1_model.pkl')

print("✅ Model saved successfully!")

✅ Model saved successfully!
